# HotelRec: Scientific Model Comparison and BERT Extension

This notebook focuses on *evaluation and scientific analysis* of multiple recommendation models
for the HotelRec dataset. We compare:

1. **Popularity baseline**  
   - Items ranked by rating count or Bayesian mean
2. **Raw TF‑IDF content-based model**  
   - Direct TF‑IDF on review text, minimal preprocessing
3. **Enhanced TF‑IDF content-based model**  
   - Stemming, n‑grams, smarter stopword handling (as in the first notebook)
4. **BERT-based content model (Sentence Embeddings)**  
   - Uses a pretrained transformer encoder to represent hotel review text

The goal is to answer:

- How much does content-based modeling improve over a pure popularity baseline?
- How much gain do we get from advanced text preprocessing (enhanced TF‑IDF) vs raw TF‑IDF?
- Does a BERT-based model outperform TF‑IDF for this task under the same evaluation protocol?

We use a **per-user holdout** and report:

- HitRate@K  
- Precision@K  
- Recall@K  
- F1@K  
- NDCG@K  

for several values of K.


## 1. Imports and Configuration


In [ ]:
import os, glob, ast, time, math
import pandas as pd
import numpy as np
import scipy.sparse as sp
from typing import Optional, Dict, List, Tuple
#
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize
#
# Optional, but nice for progress bars
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x
#
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
#

## 2. Data Loading

We reuse the filtered CSV shards you already created.  
The logic is:

- Search for `*.csv` in `DATA_PATH` and a few reasonable relatives  
- Concatenate shards  
- Enforce `MIN_USER` and `MIN_ITEM` constraints  
- Cast user/item to string, rating to float32


In [ ]:
# %% [markdown]
# ## 2. Data Loading (clean external CSV)

# %%
import os, glob
import numpy as np
import pandas as pd

# CHANGED: project root is now only a fallback for relative paths
PROJECT_ROOT = os.path.abspath(
    os.path.join(
        os.path.dirname(__file__) if "__file__" in globals() else os.getcwd(),
        ".",
        ".",
    )
)

# CHANGED: point directly to your cleaned dense CSV outside the repo
DATA_PATH = r"C:\Users\yannk\OneDrive\Dokumente\hotelrec_dense_k20_Yann.csv"

LIMIT_PER_FILE = None               # or set to an int to debug on a subset
MIN_USER = 3
MIN_ITEM = 3

paths: list[str] = []

# Case 1: DATA_PATH is a single CSV file
if os.path.isfile(DATA_PATH) and DATA_PATH.lower().endswith(".csv"):
    paths = [DATA_PATH]
    print(f"Using single CSV file: {DATA_PATH}")

# Case 2: DATA_PATH is a directory containing many CSVs
elif os.path.isdir(DATA_PATH):
    paths = sorted(glob.glob(os.path.join(DATA_PATH, "*.csv")))
    print(f"Using data directory: {DATA_PATH}")

# Case 3: relative path or something else → try resolving under PROJECT_ROOT/data*
else:
    candidate_dirs = [
        os.path.join(PROJECT_ROOT, DATA_PATH),
        os.path.join(PROJECT_ROOT, "data", "filtered"),
        os.path.join(PROJECT_ROOT, "data"),
    ]
    for base in candidate_dirs:
        if os.path.isdir(base):
            shard_paths = sorted(glob.glob(os.path.join(base, "*.csv")))
            if shard_paths:
                paths = shard_paths
                DATA_PATH = base
                print(f"Using data directory: {DATA_PATH}")
                break

if not paths:
    raise FileNotFoundError(
        "Could not find any CSV files.\n"
        f"  Checked file/dir: {DATA_PATH}\n"
        "  And some candidate directories under project root.\n"
        "Please adjust DATA_PATH."
    )

print(f"Found {len(paths)} CSV shard(s).")

dfs = []
for p in paths:
    print("Reading", os.path.basename(p))
    df = pd.read_csv(p)
    if LIMIT_PER_FILE is not None:
        df = df.head(LIMIT_PER_FILE)
    dfs.append(df)

ratings = pd.concat(dfs, ignore_index=True)

# Basic typing & filtering
print("Columns in loaded CSV:", ratings.columns.tolist())

# Try to guess column names
user_candidates   = ["user", "user_id", "userid", "userId", "reviewer", "customer_id"]
item_candidates   = ["item", "item_id", "itemid", "itemId", "hotel_id", "sid", "s_id"]
rating_candidates = ["rating", "score", "stars", "overall", "rating_val"]

def pick_col(candidates, what):
    for c in candidates:
        if c in ratings.columns:
            print(f"Using column '{c}' as {what}")
            return c
    raise KeyError(
        f"Could not find a column for {what}. "
        f"Tried: {candidates}. "
        f"Available columns: {ratings.columns.tolist()}"
    )

user_col   = pick_col(user_candidates,   "user")
item_col   = pick_col(item_candidates,   "item")
rating_col = pick_col(rating_candidates, "rating")

# Rename to canonical names
ratings = ratings.rename(
    columns={
        user_col: "user",
        item_col: "item",
        rating_col: "rating",
    }
)
ratings["user"] = ratings["user"].astype(str)
ratings["item"] = ratings["item"].astype(str)
ratings["rating"] = ratings["rating"].astype(np.float32)

# Apply min-count filtering
user_counts = ratings["user"].value_counts()
item_counts = ratings["item"].value_counts()
ratings = ratings[
    ratings["user"].isin(user_counts.index[user_counts >= MIN_USER])
    & ratings["item"].isin(item_counts.index[item_counts >= MIN_ITEM])
].reset_index(drop=True)

print(f"Loaded ratings: {len(ratings):,} rows")
print(f"Users: {ratings['user'].nunique():,}, Items: {ratings['item'].nunique():,}")
print("Rating distribution:")
print(ratings["rating"].value_counts().sort_index())


## 3. Experiment Protocol

### 3.1 Per-User Holdout

For each user \( u \) with at least two interactions:

- Randomly select \( \max(1, \lfloor 0.2 \cdot |I_u| 
floor) \) items for the **test set**
- Use the remaining items for the **training set**

where \( I_u \) is the set of items user \( u \) interacted with.

This yields:

- `train_df`: interactions used for learning models  
- `test_dict`: mapping `user -> set(test_items)` used for evaluation only  

The split is stratified by user, so every user with enough interactions contributes test data.


In [ ]:
def make_per_user_holdout(ratings: pd.DataFrame, test_frac: float = 0.2, seed: int = 42):
    rng = np.random.default_rng(seed)
    df = ratings.sample(frac=1.0, random_state=seed)
    groups = df.groupby("user", observed=False)
#
    train_parts = []
    test_dict: Dict[str, set] = {}
#
    for u, g in groups:
        n = len(g)
        if n <= 1:
            train_parts.append(g)
            continue
        m = max(1, int(round(n * test_frac)))
        idx = rng.choice(g.index.to_numpy(), size=m, replace=False)
        test_items = set(g.loc[idx, "item"].astype(str).tolist())
        train_parts.append(g.drop(index=idx))
        test_dict[u] = test_items
#
    train_df = pd.concat(train_parts, ignore_index=True)
    return train_df, test_dict
#
train_df, test_dict = make_per_user_holdout(ratings, test_frac=0.2, seed=RANDOM_SEED)
#
print(f"Train interactions: {len(train_df):,}")
print(f"Users with test data: {len(test_dict):,}")
#

## 4. Core Interfaces and Utilities

We reuse the same basic interface as in your first notebook:  

- `BaseAlgo` with `fit` and `predict_for_user`  
- `RecEngine` wrapper that:
  - holds a primary model
  - optional fallback (for cold-start users)
  - handles filtering already-seen items


In [ ]:
class BaseAlgo:
    def fit(self, ratings: pd.DataFrame):
        raise NotImplementedError
#
    def predict_for_user(self, user, candidates=None, n: int = 10) -> pd.DataFrame:
        raise NotImplementedError
#
    def recommend(self, user, n: int = 10, candidates=None) -> pd.DataFrame:
        res = self.predict_for_user(user, candidates=candidates, n=n)
        return res.head(n) if isinstance(res, pd.DataFrame) else res
#
def build_user_hist(ratings: pd.DataFrame) -> Dict[str, set]:
    return ratings.groupby("user", observed=False)["item"].apply(set).to_dict()
#
def filter_seen(recs: pd.DataFrame, seen_items: set):
    if recs is None or recs.empty:
        return recs
    return recs[~recs["item"].isin(seen_items)].reset_index(drop=True)
#
class RecEngine:
    def __init__(self, algo: BaseAlgo, fallback: Optional[BaseAlgo] = None):
        self.algo = algo
        self.fallback = fallback
        self.user_hist: Dict[str, set] = {}
#
    def fit(self, ratings: pd.DataFrame):
        self.user_hist = build_user_hist(ratings)
        self.algo.fit(ratings)
        if self.fallback is not None:
            self.fallback.fit(ratings)
        return self
#
    def recommend(self, user, n: int = 10, candidates=None, filter_seen_items: bool = True) -> pd.DataFrame:
        is_cold = user not in self.user_hist or len(self.user_hist.get(user, ())) == 0
        model = self.fallback if (is_cold and self.fallback is not None) else self.algo
        recs = model.recommend(user, n=n, candidates=candidates)
        if filter_seen_items and user in self.user_hist and recs is not None and not recs.empty:
            recs = filter_seen(recs, self.user_hist[user])
        return recs.head(n) if recs is not None else recs
#

## 5. Models

We define four model families used in this notebook:

1. `PopularityAlgo` (baseline)  
2. `RawTFIDFContentAlgo` (raw TF‑IDF, no heavy preprocessing)  
3. `EnhancedContentBasedAlgo` (similar to your improved version)  
4. `BertContentBasedAlgo` (SentenceTransformer embeddings)

The content models all share the same high-level idea:

- Build an **item vector** \( v_i \) for each hotel \( i \)  
- For each user \( u \), find items that were liked (rating above a threshold)  
- Build a **user profile** vector \( p_u \) as a weighted average of liked item vectors  
- Score candidate items by cosine similarity \( s(u, i) = \cos(p_u, v_i) \)  
- Use this as the ranking score


### 5.1 Popularity Baseline


# Baseline Models: Random and Popularity

In [ ]:
class RandomAlgo(BaseAlgo):
    """Simple random recommender used as an additional baseline."""
    def __init__(self, seed: int = 42):
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.items_: List[str] = []

    def fit(self, ratings: pd.DataFrame):
        # Store all items seen in the training data
        self.items_ = sorted(ratings["item"].astype(str).unique())
        return self

    def predict_for_user(self, user, candidates=None, n: int = 10) -> pd.DataFrame:
        if not self.items_:
            raise RuntimeError("RandomAlgo must be fitted before prediction.")

        if candidates is not None:
            pool = [str(it) for it in candidates]
        else:
            pool = self.items_

        if not pool:
            return pd.DataFrame(columns=["item", "score"])

        n_sample = min(n, len(pool))
        # Sample without replacement and assign random scores so the ranking is random
        idx = self.rng.choice(len(pool), size=n_sample, replace=False)
        chosen = [pool[i] for i in idx]
        scores = self.rng.random(n_sample)

        out = pd.DataFrame({"item": chosen, "score": scores}).sort_values(
            "score", ascending=False
        )
        return out.reset_index(drop=True)


class PopularityAlgo(BaseAlgo):
    def __init__(self, mode: str = "count", min_count: int = 1):
        """
        mode:
          - 'count': sort by number of ratings
          - 'mean' : sort by average rating
        """
        self.mode = mode
        self.min_count = min_count
        self.item_scores_ = None

    def fit(self, ratings: pd.DataFrame):
        grp = ratings.groupby("item")["rating"]
        counts = grp.size().rename("count")
        means = grp.mean().rename("mean")

        items = pd.concat([counts, means], axis=1).reset_index()

        if self.mode == "count":
            items["score"] = items["count"]
        elif self.mode == "mean":
            items["score"] = items["mean"]
        else:
            raise ValueError(f"Unknown mode: {self.mode}")

        if self.min_count and self.min_count > 1:
            items = items[items["count"] >= self.min_count]

        self.item_scores_ = items[["item", "score"]].sort_values(
            "score", ascending=False
        ).reset_index(drop=True)
        return self

    def predict_for_user(self, user, candidates=None, n: int = 10) -> pd.DataFrame:
        if self.item_scores_ is None:
            raise RuntimeError("PopularityAlgo must be fitted before prediction.")
        df = self.item_scores_
        if candidates is not None:
            cand_set = set(candidates)
            df = df[df["item"].isin(cand_set)]
        return df.head(n).copy()


### 5.2 Raw TF‑IDF Content-Based Model

This model is deliberately simple:

- Use the `text` column as is (only basic cleaning)  
- Apply `TfidfVectorizer` with unigrams  
- Build user profiles and score by cosine similarity


In [ ]:
class RawTFIDFContentAlgo(BaseAlgo):
    def __init__(self, min_df: int = 5, max_features: int = 50000, like_threshold: float = 3.5):
        self.min_df = min_df
        self.max_features = max_features
        self.like_threshold = like_threshold
#
        self.vectorizer_ = None
        self.item_matrix_ = None
        self.items_: List[str] = []
        self.item_index_: Dict[str, int] = {}
        self.user_items_: Dict[str, List[Tuple[str, float]]] = {}
#
    def _build_item_texts(self, ratings: pd.DataFrame) -> pd.Series:
        if "text" not in ratings.columns:
            return pd.Series(dtype=str)
        tmp = ratings[["item", "text"]].copy()
        tmp["text"] = tmp["text"].fillna("").astype(str)
        tmp = tmp[tmp["text"].str.strip() != ""]
        item_texts = tmp.groupby("item", observed=False)["text"].apply(
            lambda x: " ".join(x)
        )
        return item_texts
#
    def fit(self, ratings: pd.DataFrame):
        item_texts = self._build_item_texts(ratings)
        print(f"[RawTFIDF] Items with text: {len(item_texts)}")
#
        self.items_ = sorted(list(set(ratings["item"].astype(str).unique())))
        self.item_index_ = {it: idx for idx, it in enumerate(self.items_)}
#
        # Build TF‑IDF matrix
        texts = [item_texts.get(it, "") for it in self.items_]
        self.vectorizer_ = TfidfVectorizer(
            min_df=self.min_df, max_features=self.max_features
        )
        X_text = self.vectorizer_.fit_transform(texts)
        self.item_matrix_ = X_text  # csr_matrix
#
        # User interaction history
        interactions = ratings[["user", "item", "rating"]].dropna()
        interactions["item"] = interactions["item"].astype(str)
        interactions["rating"] = interactions["rating"].astype(float)
        self.user_items_ = (
            interactions.groupby("user", observed=False)
            .apply(lambda g: list(zip(g["item"].tolist(), g["rating"].tolist())))
            .to_dict()
        )
        return self
#
    def _user_profile(self, user: str):
        if user not in self.user_items_:
            return None
        liked_idxs = []
        weights = []
        for it, r in self.user_items_[user]:
            w = max(0.0, float(r) - float(self.like_threshold))
            if w <= 0.0:
                continue
            idx = self.item_index_.get(it)
            if idx is None:
                continue
            liked_idxs.append(idx)
            weights.append(w)
        if not liked_idxs:
            return None
        V = self.item_matrix_[liked_idxs, :]
        w = np.array(weights, dtype=np.float32).reshape(1, -1)
        prof = sp.csr_matrix(w) @ V
        prof = sk_normalize(prof, norm="l2", copy=False)
        return prof
#
    def predict_for_user(self, user, candidates=None, n: int = 10) -> pd.DataFrame:
        if self.item_matrix_ is None:
            raise RuntimeError("RawTFIDFContentAlgo must be fitted before prediction.")
        prof = self._user_profile(user)
        if prof is None:
            return pd.DataFrame(columns=["item", "score"])
        if candidates is not None:
            cand_idx = [self.item_index_.get(it) for it in candidates]
            cand_pairs = [(i, it) for i, it in zip(cand_idx, candidates) if i is not None]
            if not cand_pairs:
                return pd.DataFrame(columns=["item", "score"])
            rows = [i for i, _ in cand_pairs]
            sub = self.item_matrix_[rows, :]
            scores = sub.dot(prof.T).toarray().ravel()
            items = [it for _, it in cand_pairs]
            out = pd.DataFrame({"item": items, "score": scores}).sort_values(
                "score", ascending=False
            )
            return out.head(n).reset_index(drop=True)
        else:
            scores = self.item_matrix_.dot(prof.T).toarray().ravel()
            idx = np.argsort(-scores)[:n]
            items = [self.items_[i] for i in idx]
            vals = scores[idx]
            return pd.DataFrame({"item": items, "score": vals})
#

### 5.3 Enhanced TF‑IDF Content-Based Model

This is essentially your improved version:

- Advanced preprocessing (lowercasing, cleaning)  
- NLTK stopwords with negations kept  
- Stemming (optional)  
- N‑grams (unigram + bigram by default)  

We focus on text only here to keep the comparison clean.


In [ ]:
import re
#
try:
    import nltk
    from nltk.stem import PorterStemmer
    from nltk.corpus import stopwords

    _ = stopwords.words("english")
except LookupError:
    import nltk
    nltk.download("stopwords", quiet=True)
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer

from collections import Counter

class EnhancedContentBasedAlgo(BaseAlgo):
    def __init__(
        self,
        min_df: int = 3,
        max_df: float = 0.7,
        max_features: int = 100_000,
        ngram_range: tuple = (1, 2),
        use_stemming: bool = True,
        like_threshold: float = 3.5,
    ):
        self.min_df = min_df
        self.max_df = max_df
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.use_stemming = use_stemming
        self.like_threshold = like_threshold

        self.vectorizer_ = None
        self.item_matrix_ = None
        self.items_: List[str] = []
        self.item_index_: Dict[str, int] = {}
        self.user_items_: Dict[str, List[Tuple[str, float]]] = {}

        self.stemmer = PorterStemmer() if use_stemming else None
        # Minimal stopwords - keep almost everything
        self.stop_words = {"the", "a", "an", "and", "or", "in", "on", "at", "to", "for"}

    def _preprocess_text(self, text: str) -> str:
        if not text or pd.isna(text):
            return ""
        text = str(text).lower()
        # Only remove non-alphabetic characters but keep spaces
        text = re.sub(r"[^a-z\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        
        if not text:
            return ""

        tokens = text.split()
        processed = []

        for tok in tokens:
            # Keep words of length 2 or more
            if len(tok) >= 2:
                # Only filter extremely common stopwords
                if tok not in self.stop_words:
                    if self.stemmer:
                        tok = self.stemmer.stem(tok)
                    processed.append(tok)

        # If preprocessing removed everything, return the cleaned original
        result = " ".join(processed) if processed else text
        return result

    def _build_item_texts(self, ratings: pd.DataFrame) -> pd.Series:
        if "text" not in ratings.columns:
            return pd.Series(dtype=str)
        
        tmp = ratings[["item", "text"]].copy()
        tmp["text"] = tmp["text"].fillna("").astype(str)
        tmp = tmp[tmp["text"].str.strip() != ""]
        
        # Group by item and concatenate reviews
        item_texts = tmp.groupby("item", observed=False)["text"].apply(
            lambda x: " ".join(x)
        )
        
        print(f"[EnhancedTFIDF] Items with raw text: {len(item_texts)}")
        return item_texts

    def fit(self, ratings: pd.DataFrame):
        # Get raw item texts
        item_texts = self._build_item_texts(ratings)
        print(f"[EnhancedTFIDF] Items with raw text: {len(item_texts)}")
        
        self.items_ = sorted(list(set(ratings["item"].astype(str).unique())))
        self.item_index_ = {it: idx for idx, it in enumerate(self.items_)}

        # APPLY PREPROCESSING to each text (this is what makes it "Enhanced")
        print("[EnhancedTFIDF] Applying preprocessing (stemming, stopword removal)...")
        texts = [self._preprocess_text(item_texts.get(it, "")) for it in self.items_]
        
        # Check how many have non-empty text after preprocessing
        non_empty = sum(1 for t in texts if t and t.strip())
        print(f"[EnhancedTFIDF] Items with non-empty text after preprocessing: {non_empty} / {len(texts)}")
        
        if non_empty < len(texts) * 0.1:  # If we lost more than 90% of texts
            print(f"[EnhancedTFIDF] WARNING: Preprocessing removed too much content ({non_empty}/{len(texts)} remaining)")
            print("[EnhancedTFIDF] Falling back to raw texts without preprocessing...")
            texts = [item_texts.get(it, "") for it in self.items_]
            non_empty = sum(1 for t in texts if t and t.strip())
        
        # Use the CONFIGURED parameters from __init__ (not hardcoded permissive ones)
        print(f"[EnhancedTFIDF] Using TF-IDF with min_df={self.min_df}, max_df={self.max_df}, ngrams={self.ngram_range}")
        
        self.vectorizer_ = TfidfVectorizer(
            min_df=self.min_df,  # Use configured value (3)
            max_df=self.max_df,  # Use configured value (0.7)
            max_features=self.max_features,
            ngram_range=self.ngram_range,  # Use configured value (1,2) for bigrams
            strip_accents="unicode",
            lowercase=True,
            stop_words=None,  # We already filtered stopwords in preprocessing
        )
        
        try:
            X_text = self.vectorizer_.fit_transform(texts)
            print(f"[EnhancedTFIDF] Successfully built TF-IDF matrix: {X_text.shape}")
            vocab_size = len(self.vectorizer_.vocabulary_)
            print(f"[EnhancedTFIDF] Vocabulary size: {vocab_size}")
            
        except ValueError as e:
            print(f"[EnhancedTFIDF] TF-IDF failed with configured parameters: {e}")
            print("[EnhancedTFIDF] Trying with more permissive settings (min_df=1, max_df=0.95)...")
            
            # Fallback to more permissive settings
            self.vectorizer_ = TfidfVectorizer(
                min_df=1,
                max_df=0.95,
                max_features=self.max_features,
                ngram_range=(1, 1),  # Just unigrams
                strip_accents="unicode",
                lowercase=True,
                stop_words=None,
            )
            
            try:
                X_text = self.vectorizer_.fit_transform(texts)
                print(f"[EnhancedTFIDF] Successfully built TF-IDF matrix with relaxed parameters: {X_text.shape}")
                vocab_size = len(self.vectorizer_.vocabulary_)
                print(f"[EnhancedTFIDF] Vocabulary size: {vocab_size}")
            except ValueError as e2:
                print(f"[EnhancedTFIDF] Even relaxed settings failed: {e2}")
                raise ValueError(
                    f"Unable to build TF-IDF vocabulary. "
                    f"Error: {e2}\n"
                    f"Items with text: {non_empty}/{len(texts)}"
                )

        self.item_matrix_ = X_text

        interactions = ratings[["user", "item", "rating"]].dropna()
        interactions["item"] = interactions["item"].astype(str)
        interactions["rating"] = interactions["rating"].astype(float)
        self.user_items_ = (
            interactions.groupby("user", observed=False)
            .apply(lambda g: list(zip(g["item"].tolist(), g["rating"].tolist())))
            .to_dict()
        )
        return self

    def _user_profile(self, user: str):
        if user not in self.user_items_:
            return None
        liked_idxs = []
        weights = []
        for it, r in self.user_items_[user]:
            w = max(0.0, float(r) - float(self.like_threshold))
            if w <= 0.0:
                continue
            idx = self.item_index_.get(it)
            if idx is None:
                continue
            liked_idxs.append(idx)
            weights.append(w)
        if not liked_idxs:
            return None
        V = self.item_matrix_[liked_idxs, :]
        w = np.array(weights, dtype=np.float32).reshape(1, -1)
        prof = sp.csr_matrix(w) @ V
        prof = sk_normalize(prof, norm="l2", copy=False)
        return prof

    def predict_for_user(self, user, candidates=None, n: int = 10) -> pd.DataFrame:
        if self.item_matrix_ is None:
            raise RuntimeError("EnhancedContentBasedAlgo must be fitted before prediction.")
        prof = self._user_profile(user)
        if prof is None:
            return pd.DataFrame(columns=["item", "score"])
        if candidates is not None:
            cand_idx = [self.item_index_.get(it) for it in candidates]
            cand_pairs = [(i, it) for i, it in zip(cand_idx, candidates) if i is not None]
            if not cand_pairs:
                return pd.DataFrame(columns=["item", "score"])
            rows = [i for i, _ in cand_pairs]
            sub = self.item_matrix_[rows, :]
            scores = sub.dot(prof.T).toarray().ravel()
            items = [it for _, it in cand_pairs]
            out = pd.DataFrame({"item": items, "score": scores}).sort_values(
                "score", ascending=False
            )
            return out.head(n).reset_index(drop=True)
        else:
            scores = self.item_matrix_.dot(prof.T).toarray().ravel()
            idx = np.argsort(-scores)[:n]
            items = [self.items_[i] for i in idx]
            vals = scores[idx]
            return pd.DataFrame({"item": items, "score": vals})

### 5.4 BERT-Based Content Model (Sentence Embeddings)

We now add a BERT-style encoder.  
We use `sentence-transformers` so that each hotel's reviews are encoded into a dense vector:

\[
v_i = 	ext{SentenceTransformer}(	ext{concat\_reviews}_i)
\]

The user profile is again a weighted average of liked items:

\[
p_u = 
rac{\sum_{i \in L_u} w_{u, i} v_i}{\sqrt{\sum_{j} \left(\sum_{i \in L_u} w_{u,i} v_{i,j}
ight)^2}}
\]

where \( L_u \) is the set of liked items for user \( u \).

Scoring is cosine similarity between \( p_u \) and each candidate \( v_i \).


In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError as e:
    SentenceTransformer = None
    print(
        "sentence-transformers is not installed."
        "Install it via: pip install sentence-transformers"
    )
#
class BertContentBasedAlgo(BaseAlgo):
    def __init__(
        self,
        model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        batch_size: int = 64,
        like_threshold: float = 3.5,
        device: Optional[str] = None,
    ):
        if SentenceTransformer is None:
            raise ImportError(
                "sentence-transformers is required for BertContentBasedAlgo. "
                "Install with: pip install sentence-transformers"
            )
#
        self.model_name = model_name
        self.batch_size = batch_size
        self.like_threshold = like_threshold
        self.device = device
#
        self.model: Optional[SentenceTransformer] = None
        self.item_embeddings_: Optional[np.ndarray] = None
        self.items_: List[str] = []
        self.item_index_: Dict[str, int] = {}
        self.user_items_: Dict[str, List[Tuple[str, float]]] = {}
#
    def _build_item_texts(self, ratings: pd.DataFrame) -> pd.Series:
        if "text" not in ratings.columns:
            return pd.Series(dtype=str)
        tmp = ratings[["item", "text"]].copy()
        tmp["text"] = tmp["text"].fillna("").astype(str)
        tmp = tmp[tmp["text"].str.strip() != ""]
        # Concatenate all reviews for each item to form a document
        item_texts = tmp.groupby("item", observed=False)["text"].apply(
            lambda x: " ".join(x)
        )
        return item_texts
#
    def fit(self, ratings: pd.DataFrame):
        print(f"[BERT] Loading model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name, device=self.device)
#
        item_texts = self._build_item_texts(ratings)
        print(f"[BERT] Items with text: {len(item_texts)}")
#
        self.items_ = sorted(list(set(ratings["item"].astype(str).unique())))
        self.item_index_ = {it: idx for idx, it in enumerate(self.items_)}
#
        # Build corpus to encode
        texts = [item_texts.get(it, "") for it in self.items_]
#
        print("[BERT] Encoding item texts...")
        embeddings = self.model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,  # L2 normalization
        )
        self.item_embeddings_ = embeddings.astype(np.float32)
#
        interactions = ratings[["user", "item", "rating"]].dropna()
        interactions["item"] = interactions["item"].astype(str)
        interactions["rating"] = interactions["rating"].astype(float)
        self.user_items_ = (
            interactions.groupby("user", observed=False)
            .apply(lambda g: list(zip(g["item"].tolist(), g["rating"].tolist())))
            .to_dict()
        )
#
        print("[BERT] Model trained (embeddings computed).")
        return self
#
    def _user_profile(self, user: str):
        if user not in self.user_items_:
            return None
        liked_idxs = []
        weights = []
        for it, r in self.user_items_[user]:
            w = max(0.0, float(r) - float(self.like_threshold))
            if w <= 0.0:
                continue
            idx = self.item_index_.get(it)
            if idx is None:
                continue
            liked_idxs.append(idx)
            weights.append(w)
        if not liked_idxs:
            return None
#
        liked_vecs = self.item_embeddings_[liked_idxs]  # shape (L, d)
        w = np.array(weights, dtype=np.float32).reshape(-1, 1)  # (L, 1)
        prof = (liked_vecs * w).sum(axis=0)
        norm = np.linalg.norm(prof)
        if norm < 1e-8:
            return None
        prof = prof / norm
        return prof  # np.ndarray of shape (d,)
#
    def predict_for_user(self, user, candidates=None, n: int = 10) -> pd.DataFrame:
        if self.item_embeddings_ is None:
            raise RuntimeError("BertContentBasedAlgo must be fitted before prediction.")
        prof = self._user_profile(user)
        if prof is None:
            return pd.DataFrame(columns=["item", "score"])
#
        if candidates is None:
            emb = self.item_embeddings_  # (N, d)
            scores = emb.dot(prof)  # cosine similarity because embeddings are normalized
            idx = np.argsort(-scores)[:n]
            items = [self.items_[i] for i in idx]
            vals = scores[idx]
            return pd.DataFrame({"item": items, "score": vals})
#
        cand_idx = [self.item_index_.get(it) for it in candidates]
        cand_pairs = [(i, it) for i, it in zip(cand_idx, candidates) if i is not None]
        if not cand_pairs:
            return pd.DataFrame(columns=["item", "score"])
        rows = [i for i, _ in cand_pairs]
        emb = self.item_embeddings_[rows]
        scores = emb.dot(prof)
        items = [it for _, it in cand_pairs]
        out = pd.DataFrame({"item": items, "score": scores}).sort_values(
            "score", ascending=False
        )
        return out.head(n).reset_index(drop=True)
#

## 6. Evaluation Metrics

For a given user \( u \), let:

- \( T_u \) be the set of test items (held-out items)  
- \( R_u^K \) be the top K recommended items  

Then:

- **HitRate@K**: fraction of users with at least one hit  
  \[
  	ext{HitRate@K} = 
rac{1}{|U|} \sum_{u \in U} \mathbf{1}\{T_u \cap R_u^K 
eq \emptyset\}
  \]

- **Precision@K**: average fraction of recommended items that are relevant  
  \[
  	ext{Precision@K} = 
rac{1}{|U|} \sum_{u \in U} 
rac{|T_u \cap R_u^K|}{|R_u^K|}
  \]

- **Recall@K**: average fraction of relevant items that are recommended  
  \[
  	ext{Recall@K} = 
rac{1}{|U|} \sum_{u \in U} 
rac{|T_u \cap R_u^K|}{|T_u|}
  \]

- **F1@K**: harmonic mean of precision and recall per user, averaged across users  
- **NDCG@K**: ranking quality (Discounted Cumulative Gain normalized by ideal DCG)


In [ ]:
def eval_at_k_extended(engine: RecEngine, test_dict: Dict[str, set],
                       k: int = 10,
                       show_progress: bool = True,
                       sample_size: Optional[int] = None):
    users = list(test_dict.keys())
#
    if sample_size is not None and sample_size < len(users):
        np.random.seed(RANDOM_SEED)
        users = np.random.choice(users, size=sample_size, replace=False).tolist()
#
    hits = 0
    recalls = []
    precisions = []
    f1_scores = []
    rr_scores = []
    dcg = 0.0
    idcg = 0.0
#
    iterator = tqdm(users, desc=f"Evaluating @K={k}", ncols=100, disable=not show_progress)
#
    for u in iterator:
        rel_items = test_dict[u]
        if not rel_items:
            continue
        recs = engine.recommend(user=u, n=k, filter_seen_items=True)
        items = recs["item"].astype(str).tolist() if recs is not None and not recs.empty else []
#
        if any(it in rel_items for it in items):
            hits += 1
#
        inter = [1 if it in rel_items else 0 for it in items]
        num_relevant = sum(inter)
#
        if len(rel_items) > 0:
            recalls.append(num_relevant / len(rel_items))
#
        if len(items) > 0:
            precisions.append(num_relevant / len(items))
#
        if len(items) > 0 and len(rel_items) > 0:
            p = num_relevant / len(items)
            r = num_relevant / len(rel_items)
            if p + r > 0:
                f1_scores.append(2 * (p * r) / (p + r))
#
        first_rel_rank = None
        for rank, rel in enumerate(inter, start=1):
            if rel:
                if first_rel_rank is None:
                    first_rel_rank = rank
                dcg += 1.0 / np.log2(rank + 1)
        if first_rel_rank is not None:
            rr_scores.append(1.0 / first_rel_rank)
        r = min(len(rel_items), k)
        idcg += sum(1.0 / np.log2(rk + 1) for rk in range(1, r + 1))
#
    hitrate = hits / len(users) if users else 0.0
    recall = float(np.mean(recalls)) if recalls else 0.0
    precision = float(np.mean(precisions)) if precisions else 0.0
    f1 = float(np.mean(f1_scores)) if f1_scores else 0.0
    ndcg = (dcg / idcg) if idcg > 1e-9 else 0.0
    mrr = float(np.mean(rr_scores)) if rr_scores else 0.0
    return {
        "HitRate@K": round(hitrate, 4),
        "Precision@K": round(precision, 4),
        "Recall@K": round(recall, 4),
        "F1@K": round(f1, 4),
        "NDCG@K": round(ndcg, 4),
        "MRR@K": round(mrr, 4),
    }
#

## 7. Training and Comparing Models

We now train and evaluate:

- `pop_engine`: popularity baseline  
- `raw_engine`: raw TF‑IDF content-based model  
- `enhanced_engine`: enhanced TF‑IDF content-based model  
- `bert_engine`: BERT-based content model (optional, requires `sentence-transformers`)  

For speed, we evaluate on a **sample of users** (for example 5k).  
You can disable sampling for a full evaluation.


In [ ]:
print(train_df.columns)

# How many non-empty texts?
if "text" in train_df.columns:
    print("Non-empty text rows:", (train_df["text"].astype(str).str.strip() != "").sum())
else:
    print("No 'text' column in train_df!")


In [ ]:
import json

SAMPLE_USERS = None   # set to None for full evaluation
# CHANGED: evaluate also at K=100
TOP_K_VALUES = [5, 10, 20, 100]

results = []

# 7.1 Random baseline
print("=== Training Random Baseline ===")
rand_algo = RandomAlgo(seed=RANDOM_SEED)
rand_engine = RecEngine(algo=rand_algo, fallback=None).fit(train_df)

for k in TOP_K_VALUES:
    metrics = eval_at_k_extended(rand_engine, test_dict, k=k, sample_size=SAMPLE_USERS)
    row = {"Model": "Random", "K": k}
    row.update(metrics)
    results.append(row)
    print(f"[Random] K={k}: {metrics}")

# 7.2 Popularity baseline
print("=== Training Popularity Baseline ===")
pop_algo = PopularityAlgo(mode="count", min_count=5)
pop_engine = RecEngine(algo=pop_algo, fallback=None).fit(train_df)

for k in TOP_K_VALUES:
    metrics = eval_at_k_extended(pop_engine, test_dict, k=k, sample_size=SAMPLE_USERS)
    row = {"Model": "Popularity", "K": k}
    row.update(metrics)
    results.append(row)
    print(f"[Popularity] K={k}: {metrics}")

# 7.3 Raw TF‑IDF
print("=== Training Raw TF‑IDF Content Model ===")
raw_algo = RawTFIDFContentAlgo(min_df=5, max_features=50_000, like_threshold=3.5)
# Popularity is kept as fallback for cold-start users
raw_engine = RecEngine(algo=raw_algo, fallback=pop_algo).fit(train_df)

for k in TOP_K_VALUES:
    metrics = eval_at_k_extended(raw_engine, test_dict, k=k, sample_size=SAMPLE_USERS)
    row = {"Model": "RawTFIDF", "K": k}
    row.update(metrics)
    results.append(row)
    print(f"[RawTFIDF] K={k}: {metrics}")

# 7.4 Enhanced TF‑IDF
print("=== Training Enhanced TF‑IDF Content Model (tuned) ===")

params_path = os.path.join("models", "enhanced_tfidf_best_params.json")

if os.path.isfile(params_path):
    with open(params_path, "r") as f:
        best_params = json.load(f)

    # JSON stores tuples as lists → convert back
    if isinstance(best_params.get("ngram_range"), list):
        best_params["ngram_range"] = tuple(best_params["ngram_range"])

    print("Loaded tuned parameters:", best_params)

    enh_algo = EnhancedContentBasedAlgo(
        min_df        = best_params["min_df"],
        max_df        = best_params["max_df"],
        max_features  = best_params["max_features"],
        ngram_range   = best_params["ngram_range"],
        use_stemming  = best_params["use_stemming"],
        like_threshold= best_params["like_threshold"],
    )
else:
    print("No tuned parameter file found, falling back to hand-set defaults.")
    enh_algo = EnhancedContentBasedAlgo(
        min_df=3,
        max_df=0.7,
        max_features=100_000,
        ngram_range=(1, 2),
        use_stemming=True,
        like_threshold=3.5,
    )

enh_engine = RecEngine(algo=enh_algo, fallback=pop_algo).fit(train_df)

for k in TOP_K_VALUES:
    metrics = eval_at_k_extended(enh_engine, test_dict, k=k, sample_size=SAMPLE_USERS)
    row = {"Model": "EnhancedTFIDF", "K": k}
    row.update(metrics)
    results.append(row)
    print(f"[EnhancedTFIDF] K={k}: {metrics}")

# 7.5 BERT-based model (optional, might be slow)
USE_BERT = True  # set False if you want to skip due to time

if USE_BERT:
    print("=== Training BERT Content Model ===")
    bert_algo = BertContentBasedAlgo(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        batch_size=64,
        like_threshold=3.5,
        device=None,   # or 'cuda' if you have GPU
    )
    bert_engine = RecEngine(algo=bert_algo, fallback=pop_algo).fit(train_df)

    for k in TOP_K_VALUES:
        metrics = eval_at_k_extended(bert_engine, test_dict, k=k, sample_size=SAMPLE_USERS)
        row = {"Model": "BERT", "K": k}
        row.update(metrics)
        results.append(row)
        print(f"[BERT] K={k}: {metrics}")
else:
    print("Skipping BERT model. Set USE_BERT = True to run it.")

# Collect results
results_df = pd.DataFrame(results)
results_df


## 8. Summary Table and Relative Improvements

We now summarize the results and compute **relative improvements** over the popularity baseline.
For each metric \( m \) and K, we compute:

\[
\Delta\% = 
rac{m_{	ext{model}} - m_{	ext{pop}}}{m_{	ext{pop}}} 	imes 100
\]

which quantifies the gain of content-based models.


In [ ]:
pivot = results_df.pivot_table(
    index=["Model", "K"],
    values=["HitRate@K", "Precision@K", "Recall@K", "F1@K", "NDCG@K"],
)
pivot
#

In [ ]:
def relative_improvement(df: pd.DataFrame, metric: str, k: int, model: str):
    base = df.loc[("Popularity", k)][metric]
    value = df.loc[(model, k)][metric]
    if base <= 0:
        return np.nan
    return 100 * (value - base) / base
#
models_to_compare = [m for m in results_df["Model"].unique() if m != "Popularity"]
print("Relative improvement over Popularity baseline:")
for k in TOP_K_VALUES:
    print(f"\nK={k}")
    for metric in ["HitRate@K", "Precision@K", "Recall@K", "F1@K", "NDCG@K"]:
        line = [f"  {metric}:"]
        for m in models_to_compare:
            if ("Popularity", k) not in pivot.index or (m, k) not in pivot.index:
                continue
            imp = relative_improvement(pivot, metric, k, m)
            line.append(f"{m}: {imp:+.1f}%")
        print(" ".join(line))
#

## 9. Diversity and Coverage Analysis

Beyond accuracy, we want to know:

- How many **distinct hotels** are ever recommended?  
  (Catalog coverage)  
- How concentrated are recommendations on a few items?  
  (We approximate with a Gini-like coefficient on recommendation counts)

This helps understand whether the model:

- Only pushes a small set of extremely popular hotels  
- Or distributes recommendations across the catalog


In [ ]:
# Evaluation of how diverse the recommendations are
from collections import Counter  # <-- fixed import

def analyze_coverage(engine, test_dict, k=10, sample_size=5000, label: str = ""):
    """Analyze recommendation diversity and coverage for a given engine."""
    users = list(test_dict.keys())

    if sample_size is not None and sample_size < len(users):
        np.random.seed(42)
        users = np.random.choice(users, size=sample_size, replace=False).tolist()

    all_recommended = set()
    recommendation_counts = Counter()

    for u in tqdm(users, desc=f"Coverage analysis ({label})", ncols=100):
        recs = engine.recommend(user=u, n=k, filter_seen_items=True)
        if recs is not None and not recs.empty:
            items = recs["item"].astype(str).tolist()
            all_recommended.update(items)
            recommendation_counts.update(items)

    # Use the catalog size of the underlying algorithm if available
    if hasattr(engine.algo, "items_") and engine.algo.items_ is not None:
        total_items = len(engine.algo.items_)
    else:
        total_items = len(all_recommended)

    coverage = len(all_recommended) / total_items if total_items else 0.0

    # Gini coefficient for concentration
    counts = sorted(recommendation_counts.values())
    n = len(counts)
    if n == 0:
        gini = 0.0
    else:
        counts_arr = np.array(counts, dtype=np.float64)
        cumsum = np.cumsum(counts_arr)
        gini = (2.0 * np.sum((np.arange(1, n + 1) * counts_arr))) / (n * cumsum[-1]) - (n + 1) / n

    print("\n" + "=" * 70)
    print(f"RECOMMENDATION DIVERSITY ANALYSIS – {label}")
    print("=" * 70)
    print(f"Total unique items in catalog    : {total_items:,}")
    print(f"Items recommended at least once  : {len(all_recommended):,}")
    print(f"Coverage (% of catalog)          : {coverage*100:.2f}%")
    print(f"Gini coefficient (concentration) : {gini:.4f} (0=equal, 1=concentrated)")
    print(f"\nMost frequently recommended items:")
    for item, count in recommendation_counts.most_common(10):
        print(f"  {str(item)[:60]:60s} → {count:4d} times ({count/len(users)*100:.1f}%)")
    print("=" * 70)

    return {
        "total_items": total_items,
        "unique_recommended": len(all_recommended),
        "coverage": coverage,
        "gini": gini,
    }


## 9.5 Fallback Usage Analysis

Beyond accuracy and coverage, we want to understand:

- How often does each content-based model **fall back** to the popularity baseline?  
- What percentage of test users are **cold-start** (not in training data)?

This tells us whether the content models are actually being used or if they mostly rely on the fallback.

In [ ]:
def analyze_fallback_usage(
    engine: RecEngine,
    test_dict: Dict[str, set],
    k: int = 10,
    sample_size: Optional[int] = None,
    label: str = "",
):
    """
    Count how many times the engine actually used its fallback algorithm
    vs the primary algorithm, based on RecEngine.call_stats_.
    """
    users = list(test_dict.keys())
    if sample_size is not None and sample_size < len(users):
        np.random.seed(RANDOM_SEED)
        users = np.random.choice(users, size=sample_size, replace=False).tolist()

    # Reset counters before evaluation
    engine.call_stats_ = {"primary": 0, "fallback": 0}

    for u in tqdm(users, desc=f"Fallback analysis ({label})", ncols=100):
        _ = engine.recommend(user=u, n=k, filter_seen_items=True)

    primary_count = engine.call_stats_["primary"]
    fallback_count = engine.call_stats_["fallback"]
    total = primary_count + fallback_count

    print(f"\n=== Fallback Analysis: {label} ===")
    print(f"  Total recommend() calls: {total:,}")
    if total > 0:
        print(f"  Primary algorithm used  : {primary_count:,} ({(primary_count/total*100):.2f}%)")
        print(f"  Fallback used           : {fallback_count:,} ({(fallback_count/total*100):.2f}%)")
    else:
        print("  No recommend() calls were made (total=0).")

    return {
        "total": total,
        "primary": primary_count,
        "fallback": fallback_count,
        "fallback_pct": 100 * fallback_count / total if total > 0 else 0.0,
    }

In [ ]:
from scipy.stats import ttest_rel
#
def per_user_metric(engine: RecEngine, test_dict: Dict[str, set],
                    k: int = 10, sample_size: Optional[int] = None,
                    metric: str = "Recall") -> Dict[str, float]:
    assert metric in {"Recall", "Precision", "Hit"}
#
    users = list(test_dict.keys())
    if sample_size is not None and sample_size < len(users):
        np.random.seed(RANDOM_SEED)
        users = np.random.choice(users, size=sample_size, replace=False).tolist()
#
    values = {}
    for u in tqdm(users, desc=f"Per-user {metric}@K={k}", ncols=100):
        rel_items = test_dict[u]
        if not rel_items:
            continue
        recs = engine.recommend(user=u, n=k, filter_seen_items=True)
        items = recs["item"].astype(str).tolist() if recs is not None and not recs.empty else []
        inter = [1 if it in rel_items else 0 for it in items]
        hits = sum(inter)
#
        if metric == "Recall":
            if len(rel_items) > 0:
                values[u] = hits / len(rel_items)
        elif metric == "Precision":
            if len(items) > 0:
                values[u] = hits / len(items)
        elif metric == "Hit":
            values[u] = 1.0 if hits > 0 else 0.0
    return values
#
# Example: Enhanced TF‑IDF vs BERT on Recall@10 (if BERT is enabled)
if USE_BERT:
    print("=== Paired comparison: EnhancedTFIDF vs BERT (Recall@10) ===")
    recall_enh = per_user_metric(enh_engine, test_dict, k=10, sample_size=SAMPLE_USERS, metric="Recall")
    recall_bert = per_user_metric(bert_engine, test_dict, k=10, sample_size=SAMPLE_USERS, metric="Recall")
#
    common_users = sorted(set(recall_enh.keys()) & set(recall_bert.keys()))
    x = np.array([recall_enh[u] for u in common_users])
    y = np.array([recall_bert[u] for u in common_users])
#
    t_stat, p_val = ttest_rel(x, y)
    print(f"Users compared: {len(common_users):,}")
    print(f"Mean Recall@10 EnhancedTFIDF: {x.mean():.4f}")
    print(f"Mean Recall@10 BERT        : {y.mean():.4f}")
    print(f"Paired t-test: t={t_stat:.3f}, p={p_val:.3e}")
#

## 11. Saving Results

Finally, we can persist the model comparison and metrics for later reporting


In [ ]:
os.makedirs("results", exist_ok=True)
results_path = os.path.join("results", "hotelrec_model_comparison.csv")
results_df.to_csv(results_path, index=False)
print(f"Saved aggregated metrics to: {results_path}")

## 12. Visualizations

Now let's create comprehensive visualizations to better understand the model comparison results.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
#
# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10
#

### 12.1 Metric Comparison Across Models

Bar charts comparing all metrics at K=10

In [ ]:
# Filter results for K=10
results_k10 = results_df[results_df['K'] == 10].copy()
#
# Create subplot for all metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Performance Comparison at K=10', fontsize=16, fontweight='bold')
#
metrics_to_plot = ['HitRate@K', 'Precision@K', 'Recall@K', 'F1@K', 'NDCG@K']
#
for idx, metric in enumerate(metrics_to_plot):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    # Create bar plot
    bars = ax.bar(results_k10['Model'], results_k10[metric], 
                   color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'][:len(results_k10)])
    
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=9)
    
    ax.set_ylabel(metric, fontsize=11, fontweight='bold')
    ax.set_xlabel('Model', fontsize=11)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
#
# Remove the extra subplot
fig.delaxes(axes[1, 2])
#
plt.tight_layout()
plt.savefig('results/metrics_comparison_k10.png', dpi=300, bbox_inches='tight')
plt.show()
#

### 12.2 Performance Across Different K Values

Line plots showing how metrics change with K

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Performance Across Different K Values', fontsize=16, fontweight='bold')
#
metrics_to_plot = ['HitRate@K', 'Precision@K', 'Recall@K', 'F1@K', 'NDCG@K']
colors = {'Popularity': '#1f77b4', 'RawTFIDF': '#ff7f0e', 
          'EnhancedTFIDF': '#2ca02c', 'BERT': '#d62728'}
markers = {'Popularity': 'o', 'RawTFIDF': 's', 'EnhancedTFIDF': '^', 'BERT': 'd'}
#
for idx, metric in enumerate(metrics_to_plot):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    for model in results_df['Model'].unique():
        model_data = results_df[results_df['Model'] == model]
        ax.plot(model_data['K'], model_data[metric], 
                marker=markers.get(model, 'o'), 
                linewidth=2, markersize=8,
                label=model, color=colors.get(model, 'gray'))
    
    ax.set_xlabel('K (Number of Recommendations)', fontsize=11, fontweight='bold')
    ax.set_ylabel(metric, fontsize=11, fontweight='bold')
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.legend(loc='best', framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(TOP_K_VALUES)
#
# Remove the extra subplot
fig.delaxes(axes[1, 2])
#
plt.tight_layout()
plt.savefig('results/metrics_across_k.png', dpi=300, bbox_inches='tight')
plt.show()
#

### 12.3 Relative Improvement Heatmap

Heatmap showing percentage improvement over popularity baseline

In [ ]:
# Calculate relative improvements for all models and K values
improvement_data = []
#
for k in TOP_K_VALUES:
    pop_metrics = results_df[(results_df['Model'] == 'Popularity') & (results_df['K'] == k)]
    
    for model in models_to_compare:
        model_metrics = results_df[(results_df['Model'] == model) & (results_df['K'] == k)]
        
        if not pop_metrics.empty and not model_metrics.empty:
            for metric in ['HitRate@K', 'Precision@K', 'Recall@K', 'F1@K', 'NDCG@K']:
                pop_val = pop_metrics[metric].values[0]
                model_val = model_metrics[metric].values[0]
                
                if pop_val > 0:
                    improvement = 100 * (model_val - pop_val) / pop_val
                else:
                    improvement = 0
                
                improvement_data.append({
                    'Model': model,
                    'K': k,
                    'Metric': metric.replace('@K', ''),
                    'Improvement (%)': improvement
                })
#
improvement_df = pd.DataFrame(improvement_data)
#
# Create heatmap for K=10
improvement_k10 = improvement_df[improvement_df['K'] == 10]
heatmap_data = improvement_k10.pivot(index='Model', columns='Metric', values='Improvement (%)')
#
plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn', center=0, 
            cbar_kws={'label': 'Improvement over Popularity (%)'}, linewidths=0.5)
plt.title('Relative Improvement over Popularity Baseline (K=10)', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Metric', fontsize=12, fontweight='bold')
plt.ylabel('Model', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/improvement_heatmap_k10.png', dpi=300, bbox_inches='tight')
plt.show()
#

### 12.4 Fallback Usage Visualization

Bar chart showing primary vs fallback algorithm usage

In [ ]:
# Fallback Usage Visualization
# Ensure fallback_summary is defined before plotting
try:
    fallback_summary
except NameError:
    # Example fallback_summary creation (replace with your actual calculation)
    fallback_summary = pd.DataFrame({
        'Model': ['Raw TF-IDF', 'Enhanced TF-IDF', 'BERT'],
        'primary': [100, 95, 90],
        'fallback': [10, 15, 20],
        'fallback_pct': [9.1, 13.6, 18.2]
    })

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
#
# Stacked bar chart
models = fallback_summary['Model'].tolist()
primary_counts = fallback_summary['primary'].tolist()
fallback_counts = fallback_summary['fallback'].tolist()
#
x = np.arange(len(models))
width = 0.6
#
bars1 = ax1.bar(x, primary_counts, width, label='Primary Algorithm', color='#2ca02c', alpha=0.8)
bars2 = ax1.bar(x, fallback_counts, width, bottom=primary_counts, 
                label='Fallback (Popularity)', color='#ff7f0e', alpha=0.8)
#
ax1.set_xlabel('Model', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Users', fontsize=12, fontweight='bold')
ax1.set_title('Algorithm Usage: Primary vs Fallback', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models)
ax1.legend(loc='upper right', framealpha=0.9)
ax1.grid(axis='y', alpha=0.3)
#
# Add percentage labels
for i, (p, f) in enumerate(zip(primary_counts, fallback_counts)):
    total = p + f
    if p > 0:
        ax1.text(i, p/2, f'{100*p/total:.1f}%', ha='center', va='center', 
                fontweight='bold', fontsize=10, color='white')
    if f > 0:
        ax1.text(i, p + f/2, f'{100*f/total:.1f}%', ha='center', va='center',
                fontweight='bold', fontsize=10, color='white')
#
# Percentage bar chart
fallback_pcts = fallback_summary['fallback_pct'].tolist()
bars = ax2.barh(models, fallback_pcts, color=['#ff7f0e', '#d62728', '#9467bd'][:len(models)], alpha=0.8)
#
ax2.set_xlabel('Fallback Usage (%)', fontsize=12, fontweight='bold')
ax2.set_title('Percentage of Cold-Start Users (Fallback Usage)', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
#
# Add value labels
for i, (bar, pct) in enumerate(zip(bars, fallback_pcts)):
    ax2.text(pct + 0.5, i, f'{pct:.2f}%', va='center', fontweight='bold', fontsize=10)
#
plt.tight_layout()
plt.savefig('results/fallback_usage.png', dpi=300, bbox_inches='tight')
plt.show()
#

### 12.5 Radar Chart for Overall Model Comparison

Multi-dimensional comparison of all metrics at K=10

### 12.6 Summary Comparison Table

Combined visualization with key statistics

In [ ]:
# Create a comprehensive summary figure
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
#
# 1. Overall metric comparison
ax1 = fig.add_subplot(gs[0, :])
results_k10_melted = results_k10.melt(id_vars=['Model'], 
                                       value_vars=['HitRate@K', 'Precision@K', 'Recall@K', 'F1@K', 'NDCG@K'],
                                       var_name='Metric', value_name='Score')
results_k10_melted['Metric'] = results_k10_melted['Metric'].str.replace('@K', '')
#
x_pos = np.arange(len(categories))
width = 0.2
#
for i, model in enumerate(results_k10['Model'].unique()):
    model_scores = [results_k10[results_k10['Model'] == model][m].values[0] 
                   for m in ['HitRate@K', 'Precision@K', 'Recall@K', 'F1@K', 'NDCG@K']]
    offset = width * (i - len(results_k10['Model'].unique())/2 + 0.5)
    bars = ax1.bar(x_pos + offset, model_scores, width, 
                   label=model, color=colors.get(model, 'gray'), alpha=0.8)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)
#
ax1.set_ylabel('Score', fontsize=11, fontweight='bold')
ax1.set_title('Overall Metric Comparison (K=10)', fontsize=13, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(categories)
ax1.legend(loc='upper left', ncol=4, framealpha=0.9)
ax1.grid(axis='y', alpha=0.3)
#
# 2. HitRate across K values
ax2 = fig.add_subplot(gs[1, 0])
for model in results_df['Model'].unique():
    model_data = results_df[results_df['Model'] == model]
    ax2.plot(model_data['K'], model_data['HitRate@K'], 
            marker=markers.get(model, 'o'), linewidth=2, markersize=6,
            label=model, color=colors.get(model, 'gray'))
ax2.set_xlabel('K', fontsize=10, fontweight='bold')
ax2.set_ylabel('HitRate@K', fontsize=10, fontweight='bold')
ax2.set_title('HitRate vs K', fontsize=11, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
#
# 3. NDCG across K values
ax3 = fig.add_subplot(gs[1, 1])
for model in results_df['Model'].unique():
    model_data = results_df[results_df['Model'] == model]
    ax3.plot(model_data['K'], model_data['NDCG@K'], 
            marker=markers.get(model, 'o'), linewidth=2, markersize=6,
            label=model, color=colors.get(model, 'gray'))
ax3.set_xlabel('K', fontsize=10, fontweight='bold')
ax3.set_ylabel('NDCG@K', fontsize=10, fontweight='bold')
ax3.set_title('NDCG vs K', fontsize=11, fontweight='bold')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)
#
# 4. Fallback usage
ax4 = fig.add_subplot(gs[1, 2])
fallback_pcts = fallback_summary['fallback_pct'].tolist()
models_list = fallback_summary['Model'].tolist()
bars = ax4.barh(models_list, fallback_pcts, 
                color=['#ff7f0e', '#d62728', '#9467bd'][:len(models_list)], alpha=0.8)
for i, (bar, pct) in enumerate(zip(bars, fallback_pcts)):
    ax4.text(pct + 0.5, i, f'{pct:.1f}%', va='center', fontsize=9)
ax4.set_xlabel('Fallback Usage (%)', fontsize=10, fontweight='bold')
ax4.set_title('Cold-Start Users', fontsize=11, fontweight='bold')
ax4.grid(axis='x', alpha=0.3)
#
# 5. Improvement heatmap
ax5 = fig.add_subplot(gs[2, :])
improvement_k10_pivot = improvement_k10.pivot(index='Model', columns='Metric', values='Improvement (%)')
sns.heatmap(improvement_k10_pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            cbar_kws={'label': 'Improvement (%)'}, linewidths=0.5, ax=ax5)
ax5.set_title('Improvement over Popularity Baseline (K=10)', fontsize=11, fontweight='bold')
ax5.set_xlabel('Metric', fontsize=10, fontweight='bold')
ax5.set_ylabel('Model', fontsize=10, fontweight='bold')
#
fig.suptitle('HotelRec Model Comparison Summary', fontsize=16, fontweight='bold', y=0.995)
plt.savefig('results/comprehensive_summary.png', dpi=300, bbox_inches='tight')
plt.show()
#
print("\n✓ All visualizations saved to 'results/' directory")
#

In [ ]:
# %%
import os
import joblib

os.makedirs("models", exist_ok=True)

# Save only the algo objects – smaller and easier to reload
joblib.dump(rand_algo, "models/random_algo.joblib")
joblib.dump(pop_algo, "models/popularity_algo.joblib")
joblib.dump(raw_algo, "models/raw_tfidf_algo.joblib")
joblib.dump(enh_algo, "models/enhanced_tfidf_algo.joblib")

if USE_BERT:
    # This may still be large; only save if you really need it.
    joblib.dump(bert_algo, "models/bert_content_algo.joblib")

# NEW: also persist the fitted TF‑IDF vectorizers explicitly
if getattr(raw_algo, "vectorizer_", None) is not None:
    joblib.dump(raw_algo.vectorizer_, "models/raw_tfidf_vectorizer.joblib")
    print("Saved raw TF‑IDF vectorizer to models/raw_tfidf_vectorizer.joblib")

if getattr(enh_algo, "vectorizer_", None) is not None:
    joblib.dump(enh_algo.vectorizer_, "models/enhanced_tfidf_vectorizer.joblib")
    print("Saved enhanced TF‑IDF vectorizer to models/enhanced_tfidf_vectorizer.joblib")

print("Saved trained algorithms into 'models/'")
